# <font color='#000000'>__Exercício Prático 2__</font>
## <font color='#e09422'>Dashboard Tableau</font>
#### <font color='#4b4b4b'>Master Big Data Aplicado ao Futebol <br> Módulo 8 - Ferramentas de Visualização - Tableau e PowerBI <br> Desenvolvido por: Hugo Alves </font>

__Enunciado do Exercício__

<i> Usando os dados disponíveis dos Feeds F24 da OPTA (ou outros de vossa preferência) pretende-se que sejam capazes de gerar uma dashboard onde expliquem ao vosso treinador um momento de jogo ou característica da próxima equipa com quem vão jogar. Tenham em conta que, por norma (nem sempre se aplicará), é interessante saber:
- O quê é que esta equipa faz
- Quem são os principais executantes/responsáveis a nível individual
- Onde, no campo, acontece mais vezes
- Evolução dessa tendência/característica ao longo dos últimos jogos.

Tenham em conta também que a dashboard deve ser visualmente atrativa para vos ajudar a captar a atenção do treinador e que demasiada informação muito “apertada” na maioria das vezes transforma-se em ruido. </i>
<br><br>
__Nota 1:__ Devido ao espaço ocupado pelos ficheiros XML com os dados da OPTA, este notebook foi preparado para correr em Google Colab.

__Nota 2:__ A essência do código foi aproveitado do Exercício Prático 2 do módulo 5 - Análise de dados no futebol com Python. Neste notebook, o objetivo será transformar os dados brutos dos feeds F24 da OPTA para obter um dataset próprio para exploração através de um dashboard em Tableau, e cujo objetivo será avaliar a saída em construção de uma determinada equipa.

# <font color="#e09422">__________________</font>
## <font color="#4b4b4b">Índice de Conteúdos</font> <a class="anchor" id="toc"></a>
[1. Setup Inicial](#setup)<br>
- [1.1. Packages e Funções](#pack)<br>
- [1.2. Ficheiros de Eventos](#event)<br>

[2. Processamento de Eventos](#process)<br>

# <font color="#e09422">____________</font>
## <font color='#4b4b4b'>1. Setup Inicial</font> <a class="anchor" id="setup"></a>
[Regressar ao Índice](#toc)

### <font color='#e09422'>1.1. Packages e Funções</font> <a class="anchor" id="pack"></a>
[Regressar ao Índice](#toc)

In [1]:
!python --version

Python 3.12.13


Vamos utilizar a versão 3.12.13 do Python.

In [2]:
import os
import xml.etree.ElementTree as ET
import time
from tqdm.notebook import tqdm
from typing import Optional
from google.colab import files
import zipfile

import numpy as np
import pandas as pd

import requests
from bs4 import BeautifulSoup

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

Na célula seguinte encontram-se as funções utilizadas ao longo deste notebook. O ideal seria ter um ficheiro .py separado com as funções, mas neste caso, em que estamos a trabalhar num ficheiro colaborativo, talvez seja mais prático ter tudo junto.

In [3]:
# Função auxiliar para mover colunas
def move_column(df: pd.DataFrame,
                col_to_move: str,
                after_col: str) -> pd.DataFrame:
  """
  Move uma coluna para uma posição específica de um DataFrame.

  Parâmetros:
  - df (pd.DataFrame): DataFrame a alterar.
  - col_to_move (str): Coluna a mover.
  - after_col (str): Coluna que deve anteceder a coluna movida.

  Devolve:
  - pd.DataFrame: DataFrame com a coluna movida.
  """
  cols = list(df.columns)
  cols.remove(col_to_move)
  insert_at = cols.index(after_col) + 1
  cols.insert(insert_at, col_to_move)
  return df[cols]

# Função para extrair dados de uma pasta com ficheiros XML para um DataFrame
def parse_F24_folder(folder_path: str,
                     event_descriptions: pd.DataFrame,
                     player_descriptions: Optional[pd.DataFrame] = None) -> pd.DataFrame:
  """
  Recebe uma pasta com ficheiros XML de dados da OPTA (F24) e devolve um DataFrame com os dados transformados.
  Além de trazer os dados para um DataFrame único, aplica transformações sobre as colunas, data types, e
  complementa com a descrição dos eventos e jogadores que os executaram (este último opcional).

  Parâmetros:
  - folder_path (str): Caminho para a pasta com os ficheiros XML.
  - event_descriptions (pd.DataFrame): DataFrame com a descrição dos eventos. Deve conter as colunas "type_id" e "event_name".
  - player_descriptions (pd.DataFrame): DataFrame com a descrição dos jogadores.

  Devolve:
  - pd.DataFrame: DataFrame com os dados transformados.
  """
  t0 = time.perf_counter()
  games_list = []
  events_list = []

  ### 1. Carregar ficheiros XML
  for file in tqdm(os.listdir(folder_path)):
    # Ignorar ficheiros que não XML
    if file.endswith(".xml"):
      file_path = os.path.join(folder_path, file)
      tree = ET.parse(file_path)
      games = tree.getroot()
      # Assumindo que cada ficheiro diz respeito a um jogo, vamos buscar os metadados dessa partida
      game_info = games.find("Game")

      game_id = game_info.get("id")
      game_meta = {
        "game_id": game_id,
        "season_id": game_info.get("season_id"),
        "season_name": game_info.get("season_name"),
        "competition_id": game_info.get("competition_id"),
        "competition_name": game_info.get("competition_name"),
        "matchday": game_info.get("matchday"),
        "game_date": game_info.get("game_date"),
        "home_team_id": game_info.get("home_team_id"),
        "home_team_name": game_info.get("home_team_name"),
        "home_score": game_info.get("home_score"),
        "away_team_id": game_info.get("away_team_id"),
        "away_team_name": game_info.get("away_team_name"),
        "away_score": game_info.get("away_score"),
      }
      games_list.append(game_meta)

      # Iterar sobre todos os eventos da partida
      for game in games:
        for event in game.findall("Event"):
          event_data = event.attrib.copy()
          event_data["game_id"] = game_id
          # Iterar sobre os qualifiers (identificados por "Q" nos ficheiros XML)
          event_data["qualifiers"] = [q.attrib for q in event.findall("Q")]
          events_list.append(event_data)

  print("Carregamento dos ficheiros XML concluído. A iniciar transformação do DataFrame")
  t1 = time.perf_counter()
  print(f"Duração do carregamento dos ficheiros XML: {t1 - t0:.2f}s")

  ### 2. Juntar jogos e eventos
  df_games = pd.DataFrame(games_list)
  df_events = pd.DataFrame(events_list)

  # Associar metadados dos jogos a cada evento
  df_events = pd.merge(df_events, df_games, how = "left", on = "game_id")

  ### 3. Transformar data types
  # Colunas numéricas (números inteiros)
  small_numeric_cols = ["event_id", "type_id", "period_id", "min", "sec", "team_id", "season_id", "competition_id",
                        "matchday", "home_team_id", "home_score", "away_team_id", "away_score"]
  medium_numeric_cols = ["game_id", "player_id"]
  big_numeric_cols = ["id", "version"]
  for col in small_numeric_cols:
    df_events[col] = pd.to_numeric(df_events[col], errors = "coerce").astype("Int16")
  for col in medium_numeric_cols:
    df_events[col] = pd.to_numeric(df_events[col], errors = "coerce").astype("Int32")
  for col in big_numeric_cols:
    df_events[col] = pd.to_numeric(df_events[col], errors = "coerce").astype("Int64")

  # Colunas numéricas (coordenadas com casas decimais)
  df_events["x"] = pd.to_numeric(df_events["x"], errors = "coerce").astype("Float32")
  df_events["y"] = pd.to_numeric(df_events["y"], errors = "coerce").astype("Float32")

  # Colunas booleanas (0 ou 1s) - nestes casos, será seguro assumir que a ausência de valor representa 0
  boolean_cols = ["outcome", "keypass", "assist"]
  for col in boolean_cols:
    # vamos utilizar o data type numérico em vez de boolean
    df_events[col] = pd.to_numeric(df_events[col], errors = "coerce").fillna(0).astype("Int8")

  # Datas
  df_events["timestamp"] = pd.to_datetime(df_events["timestamp"], errors = "coerce")
  df_events["last_modified"] = pd.to_datetime(df_events["last_modified"], errors = "coerce")
  df_events["game_date"] = pd.to_datetime(df_events["game_date"], errors = "coerce")

  # Colunas categóricas (colunas com relativamente poucos valores únicos face ao total de linhas)
  category_cols = ["season_name", "competition_name", "home_team_name", "away_team_name"]
  for col in category_cols:
    df_events[col] = df_events[col].astype("category")

  ### 4. Juntar descrição dos eventos, qualifiers e jogadores
  try:
    df_events = pd.merge(df_events, event_descriptions, how = "left", on = "type_id")
    # Mover a descrição para depois do tipo de evento
    df_events = move_column(df_events, "event_name", "type_id")
  except Exception:
    print("event_descriptions deve ser um DataFrame e conter as colunas 'type_id' e 'event_name'.")
    return

  if player_descriptions is not None:
    if isinstance(player_descriptions, pd.DataFrame) and "player_id" in player_descriptions.columns:
      # Guardar nome das colunas para adicionar prefixo às colunas novas (a função pd.merge só aceita sufixos)
      existing_cols = set(df_events.columns)
      df_events = pd.merge(df_events, player_descriptions, how = "left", on = "player_id")
      # Adicionar prefixo e mover as colunas para imediatamente depois do player_id
      new_cols = list(set(df_events.columns) - existing_cols - {"player_id"})
      new_cols_renamed = [f"player_{col}" for col in new_cols]
      df_events.rename(columns = dict(zip(new_cols, new_cols_renamed)), inplace = True)
      for col in new_cols_renamed:
        df_events = move_column(df_events, col, "player_id")
    else:
      print("player_descriptions deve ser um DataFrame e conter a coluna 'player_id'.")

  t2 = time.perf_counter()
  print("=====================================================================================")
  print(f"Transformação do DataFrame concluída em {t2 - t1:.2f}s")
  print(f"Duração total: {t2 - t0:.2f}s\n")
  return df_events

# "Explode" os eventos de um DataFrame
def explode_events(df_events: pd.DataFrame,
                   event_type: int,
                   qualifier_descriptions: Optional[pd.DataFrame] = None) -> pd.DataFrame:
  """
  "Explode" os eventos de um DataFrame. Recebe um DataFrame em que cada linha corresponde a um evento,
  e devolve um DataFrame em que cada linha passa a ser uma combinação evento & qualifier.

  Parâmetros:
  - df_events (pd.DataFrame): DataFrame com os eventos.
  - event_type (int): ID do evento a "explodir".
  - qualifier_descriptions (pd.DataFrame): DataFrame com a descrição dos qualifiers.
  """
  print("A iniciar transformação")
  t0 = time.perf_counter()

  # Criar cópia do DataFrame e validar a existência de eventos
  df = df_events[df_events["type_id"] == event_type].copy()
  if df.empty:
    print(f"Não foram encontrados eventos com o ID {event_type}.")
    return pd.DataFrame()

  # "Explodir" qualifiers (partindo do princípio que é uma lista de dicionários)
  df_exploded = df.explode("qualifiers")

  # Normalizar qualifiers
  df_q = pd.json_normalize(df_exploded["qualifiers"]).fillna("Yes")
  df_q["id"] = df_exploded["id"].values

  # Fazer pivot dos qualifiers. Para cada ID, cada qualifier passa a ser uma coluna (com o respetivo valor)
  df_q = df_q.pivot_table(index = "id", columns = "qualifier_id", values = "value", aggfunc = "first").reset_index()

  # Associar descrição dos qualifiers (se o parâmetro tiver sido passado)
  if qualifier_descriptions is not None:
    if isinstance(qualifier_descriptions, pd.DataFrame) and {"qualifier_id", "description"}.issubset(qualifier_descriptions.columns):
      dict_q = dict(zip(qualifier_descriptions["qualifier_id"].astype(str), qualifier_descriptions["description"].astype(str)))
      df_q.rename(columns = dict_q, inplace = True)
    else:
      print("qualifier_descriptions deve ser um DataFrame e conter as colunas 'qualifier_id' e 'description'.\nA continuar transformação sem a descrição dos qualifiers.")

  # Voltar a juntar os DataFrames e preencher qualifiers em falta com "-"
  qualifier_cols = df_q.columns.difference(["id"])
  df = df.drop(columns = ["qualifiers"]).merge(df_q, on = "id", how = "left")
  df[qualifier_cols] = df[qualifier_cols].fillna("-")

  t1 = time.perf_counter()
  print("=====================================================================================")
  print(f"Duração da transformação dos eventos: {t1 - t0:.2f}s\n")

  return df

# Função para definir a direção de um passe
def classify_direction(angle_rad: float) -> str:
  """
  Recebe o ângulo de um passe (em radianos) e converte para uma de 8 direções.

  Parâmetros:
  - angle_rad (float): Ângulo do passe em radianos.

  Devolve:
  - str: Direção do passe.
  """
  deg = np.degrees(angle_rad) % 360

  if deg <= 30 or deg > 330:
      return "Frente"
  elif 30 < deg <= 80:
      return "Frente-Direita"
  elif 80 < deg <= 100:
      return "Direita"
  elif 100 < deg <= 150:
      return "Trás-Direita"
  elif 150 < deg <= 210:
      return "Trás"
  elif 210 < deg <= 260:
      return "Trás-Esquerda"
  elif 260 < deg <= 280:
      return "Esquerda"
  elif 280 < deg <= 330:
      return "Frente-Esquerda"

### <font color='#e09422'>1.2. Ficheiros de Eventos</font> <a class="anchor" id="event"></a>
[Regressar ao Índice](#toc)

Vamos começar por ir buscar a pasta ZIP com os dados providenciados pela OPTA.

In [4]:
folder_name = "OPTA Data"

In [5]:
if os.path.isdir(folder_name):
  print(f"A pasta '{folder_name}' já existe. A usar os ficheiros existentes.")
else:
  print(f"Pasta não encontrada. Necessário fazer upload do ficheiro ZIP.")

  uploaded = files.upload()

  zip_name = list(uploaded.keys())[0]
  with zipfile.ZipFile(zip_name, "r") as zip_ref:
      zip_ref.extractall(folder_name)

Pasta não encontrada. Necessário fazer upload do ficheiro ZIP.


Saving OPTA Data.zip to OPTA Data.zip


In [6]:
folder_path = "OPTA Data/OPTA Data/F24 - Portugal"

Vamos agora importar dois ficheiros auxiliares com as descrições dos IDs dos eventos e qualifiers. Estes ficheiros foram criados tendo por base os dicionários do ficheiro `parse_24` fornecido junto com o enunciado, portanto fica o agradecimento pelo trabalho poupado.

Por sua vez, este dicionário terá por base a documentação da OPTA, partilhada via PDF e disponível [online](https://github.com/jokecamp/FootballData/blob/master/random_docs/Opta-f24_appendices.docx).

In [7]:
df_event_types = pd.read_csv("OPTA Data/OPTA Data/Event Types.csv")
df_qualifiers = pd.read_csv("OPTA Data/OPTA Data/Qualifier Descriptions.csv")
df_players = pd.read_excel("OPTA Data/OPTA Data/opta_planteis_portugal.xlsx")

De seguida, chamamos a função para extrair os dados dos ficheiros XML fornecidos. Esta função é também adaptada do ficheiro `parse_24`, embora com algumas diferenças (especialmente no que respeita ao processamento do DataFrame dos eventos).

In [8]:
df_events = parse_F24_folder(folder_path,
                             event_descriptions = df_event_types,
                             player_descriptions = df_players)

  0%|          | 0/300 [00:00<?, ?it/s]

Carregamento dos ficheiros XML concluído. A iniciar transformação do DataFrame
Duração do carregamento dos ficheiros XML: 23.27s
Transformação do DataFrame concluída em 12.21s
Duração total: 35.48s



Perfeito. Agora que já temos um dataset completo e transformado com os eventos da partida e informação complementar, podemos prosseguir para a definição das nossas métricas.

# <font color="#e09422">_________________________</font>
## <font color='#4b4b4b'>2. Processamento de Eventos</font> <a class="anchor" id="process"></a>
[Regressar ao Índice](#toc)

Começamos desde já por criar variáveis auxiliares que poderão ser importantes para incluir no dashboard, e que permitirão saber se a equipa se encontrava ou não em desvantagem à data do evento. A melhor forma de o fazer será contar, para um dado jogo, os eventos de golos que tinham ocorrido para cada equipa até esse momento.

__Nota__: Por uma questão de consistência, vamos continuar a usar os nomes das variáveis em inglês.

In [9]:
# Começamos por ordenar o dataset por jogo, parte, minuto, segundo, timestamp, e ID do evento (no limite para desempatar)
df_events = df_events.sort_values(["game_id", "period_id", "min", "sec", "timestamp", "id"]).reset_index(drop = True)

# Variáveis auxiliares para identificar golos das equipas (ID 16, segundo o apêndice da OPTA)
df_events["home_goal"] = ((df_events["type_id"] == 16) & (df_events["team_id"] == df_events["home_team_id"])).astype("Int8")
df_events["away_goal"] = ((df_events["type_id"] == 16) & (df_events["team_id"] == df_events["away_team_id"])).astype("Int8")

# Somar cumulativamente os eventos de golo de cada equipa dentro de uma partida
df_events["home_score_current"] = df_events.groupby("game_id")["home_goal"].cumsum()
df_events["away_score_current"] = df_events.groupby("game_id")["away_goal"].cumsum()

# Variável para identificar se a equipa está a perder
df_events["is_losing"] = (
    ((df_events["team_id"] == df_events["home_team_id"]) & (df_events["home_score_current"] < df_events["away_score_current"])) \
    | \
    ((df_events["team_id"] == df_events["away_team_id"]) & (df_events["away_score_current"] < df_events["home_score_current"]))
).astype("Int8")

# Apagar colunas auxiliares que temos a certeza que não serão mais utilizadas
df_events.drop(columns = ["home_goal", "away_goal"], inplace = True)

Vamos usar a função `explode_events` para converter os qualifiers no formato tabular com que estamos a trabalhar.

In [10]:
df_passes = explode_events(df_events, 1, df_qualifiers)

A iniciar transformação
Duração da transformação dos eventos: 18.50s



Vamos desde já eliminar as colunas cujos qualifiers não estão mapeados. Não sabendo o que significam, não serão certamente utilizadas em qualquer tipo de análise no Tableau.

In [11]:
df_passes.drop(columns = ["152", "233", "236", "237", "238", "240", "241", "278", "279", "286", "287", "343", "345", "362", "386", "387", "388", "389"], inplace = True)

No módulo 5, observámos no Apêndice 3 dos ficheiros F24 da OPTA ("useful queries") que, para a contabilização de passes, deveriam ser eliminados os que tivessem os qualifiers 2, 5, 6, 107, 123, e 124. Contudo, uma vez que agora não estamos interessados em obter estatísticas mas sim em analisar padrões de construção e progressão no terreno, não faz sentido eliminar estes passes (até porque qualifiers como "Keeper Throw" e "Goal Kick" poderão ter agora outro tipo de relevância).

Vamos, isso sim, aproveitar para converter a coluna `length` para um formato numérico e converter esta distância para metros, visto que a descrição deste qualifier (ID 212) nos apêndices indica que este valor está em jardas.

In [12]:
df_passes["Length"] = pd.to_numeric(df_passes["Length"], errors = "coerce")

# Segundo a internet, 1 jarda = 0.9144 metros
df_passes["Length"] = df_passes["Length"] * 0.9144

De seguida, podemos aproveitar para calcular outra coluna para obter o nome da equipa que fez o passe.

In [13]:
df_passes["team_name"] = np.where(
  df_passes["team_id"] == df_passes["home_team_id"],
  df_passes["home_team_name"],
  df_passes["away_team_name"]
)

Podemos também acrescentar uma variável para identificar os últimos jogos de cada equipa: por exemplo, o jogo mais recente terá o valor 1, o segundo mais recente 2, e assim por diante.

In [14]:
df_passes["game_rank"] = (
    df_passes.sort_values(["team_id", "game_date"], ascending = [True, False])
    .groupby("team_id")["game_id"]
    .transform(lambda x: x.map({gid: rank+1 for rank, gid in enumerate(x.unique())}))
)

Num cenário de aplicação como este, em que o objetivo concreto é avaliar a construção e progressão de uma equipa através dos passes, vamos __limitar os passes aos ocorridos antes do último terço__ (e reduzindo assim a extensa lista de eventos que temos de momento). Depois, no dashboard, a ideia será disponibilizar um filtro para permitir ao utilizador avaliar zonas mais específicas do terreno.

Nessa sequência, podemos também criar uma variável simples para definir se o passe ocorreu em zona de construção ou de criação. Esta definição pode variar muito em função do estilo de jogo de cada equipa - há equipas em que a zona de criação poderá ser o terço intermédio do campo, outras em que pode ser entre o seu meio-campo e a entrada da área contrária (algo que já nem iríamos captar por estarmos a filtrar por passes antes do último terço). Ainda assim, vamos entender a zona de construção como o primeiro terço e a zona de criação como o segundo. Num contexto real, __esta seria uma interação a ter com a equipa técnica, para respeitar o seu entendimento e interpretação de cada zona__.

In [15]:
df_passes = df_passes[df_passes["x"] < 66.67]
df_passes["pitch_zone"] = df_passes["x"].apply(lambda x: "Zona de Construção" if x < 33.33 else "Zona de Criação")

Outra variável menos "controversa" mas igualmente útil será a `corridor`, para identificar o corredor onde foi feito o passe.

In [16]:
df_passes["corridor"] = df_passes["y"].apply(lambda y: "Direito" if y < 33.33 else ("Esquerdo" if y > 66.67 else "Centro"))

A [OPTA](https://theanalyst.com/articles/opta-football-stats-definitions) define passes progressivos como "passes nos dois últimos terços atacantes que levam a bola pelo menos 25% mais próximo da baliza". Podemos criar uma variável auxiliar para identificar estes passes.

__Nota:__ O facto de estarmos a acrescentar todas estas novas colunas não significa que estas venham a ser todas utilizadas no dashboard. O objetivo é oferecer maior flexibilidade e possibilidades na criação dos visuais, abrindo mais margem para evoluções futuras à medida dos utilizadores.

In [17]:
# Necessário converter as coordenadas do fim dos passes para valores numéricos, algo que ainda não fizemos
df_passes["Pass End X"] = pd.to_numeric(df_passes["Pass End X"], errors = "coerce")
df_passes["Pass End Y"] = pd.to_numeric(df_passes["Pass End Y"], errors = "coerce")
print(df_passes[["Pass End X", "Pass End Y"]].isna().sum())

Pass End X    0
Pass End Y    0
dtype: int64


In [18]:
# Distância (euclidiana) até à baliza adversária (assumimos o centro da baliza contrária onde x = 100 e y = 50)
df_passes["dist_to_goal_start"] = np.sqrt((100 - df_passes["x"]) ** 2 + (50 - df_passes["y"]) ** 2)
df_passes["dist_to_goal_end"] = np.sqrt((100 - df_passes["Pass End X"]) ** 2 + (50 - df_passes["Pass End Y"]) ** 2)

# Passe progressivo
df_passes["is_progressive_pass"] = (
    (df_passes["x"] >= 33.33) &
    (df_passes["outcome"] == 1) &
    (df_passes["dist_to_goal_end"] <= df_passes["dist_to_goal_start"] * 0.75)
)

# Eliminar colunas auxiliares
df_passes.drop(columns = ["dist_to_goal_start", "dist_to_goal_end"], inplace = True)

Por fim, vamos também simplificar a variável de ângulo de passe que temos atualmente para obter uma de 8 direções possíveis.

In [19]:
df_passes["Angle"] = pd.to_numeric(df_passes["Angle"], errors = "coerce")

df_passes["pass_direction"] = df_passes["Angle"].apply(classify_direction)

Podemos agora exportar os dados para um ficheiro Excel, que depois alimentará o Tableau. Num cenário de produção, __o recomendado seria ter todo este código num script .py__ para receber a pasta ZIP com os dados e gerar automaticamente o ficheiro como output. Aqui, optámos por ter um notebook para poder melhor estruturar a explicação dos passos e decisões que tomámos.

Outro ponto importante é que, neste momento, __vamos exportar dados de todas as equipas do campeonato__. No dashboard, apenas vamos analisar uma equipa de cada vez, até porque o cenário colocado envolve estudar o próximo adversário (definiremos quem vai ser no Tableau). Ainda assim, esta decisão é justificada para permitir ao utilizador mais flexibilidade e menos dependência na análise, além de oferecer a possibilidade de comparar diferentes equipas de forma muito mais simples e direta.

__Nota do futuro:__ Apesar da intenção expressa acima, limitações computacionais estavam a comprometer a capacidade do Tableau de processar a informação em tempo útil. Assim, e uma vez que o objetivo proposto passa também por apresentar um dashboard que permita retirar conclusões concretas e imediatas, sem necessidade da aplicação de filtros, __vamos optar por exportar dados dos últimos 10 jogos de uma única equipa__. Nesse sentido, e simplesmente pelo facto de, na data de fim do módulo (04/05/2026) o jogo seguinte do Sporting CP ser frente ao Rio Ave FC, vamos selecionar este adversário como a equipa em foco na nossa análise.

In [20]:
print("Nº de linhas pré-filtragem:", df_passes.shape[0])

TEAM_NAME = "Rio Ave"
NUMBER_MATCHES = 10

df_passes = df_passes[(df_passes["team_name"] == TEAM_NAME) & (df_passes["game_rank"] <= NUMBER_MATCHES)]

print("Nº de linhas pós-filtragem:", df_passes.shape[0])

Nº de linhas pré-filtragem: 222186
Nº de linhas pós-filtragem: 3826


In [21]:
df_passes.to_excel("Passes Construção e Progressão - Completo.xlsx", index = False)

Vamos gerar um ficheiro auxiliar para mapear as zonas do campo definidas diretamente no Tableau (também o podíamos ter feito aqui, mas a ideia foi mesmo explorar as funcionalidades do Tableau). Esta tabela vai ser útil para definir os limites de cada zona a apresentar no pitch map.

__Nota:__ Para acelerar a criação deste ficheiro, o código para definir as zonas em Tableau foi partilhado com o Claude, que gerou o código Python com os limites correspondentes.

In [22]:
x_limits = [0, 17, 33.33, 50, 66.67]
y_limits = [0, 33.33, 66.67, 100]

rows = []
zone_id = 1
for i in range(len(x_limits) - 1):
    for j in range(len(y_limits) - 1):
        x_min, x_max = x_limits[i], x_limits[i+1]
        y_min, y_max = y_limits[j], y_limits[j+1]
        corners = [
            (x_min, y_min, 1),
            (x_max, y_min, 2),
            (x_max, y_max, 3),
            (x_min, y_max, 4),
        ]
        for x, y, point_order in corners:
            rows.append({
                "zone_id": zone_id,
                "x": x,
                "y": y,
                "point_order": point_order
            })
        zone_id += 1

df_zones = pd.DataFrame(rows)
df_zones.to_excel("Pitch Zones Tableau.xlsx", index = False)

Para terminar, vamos definir as posições de cada direção de passe numa grelha 3x3, para depois aplicar num outro visual em Tableau.

In [23]:
grid = [
    {"pass_direction": "Frente-Esquerda",  "grid_x": 1, "grid_y": 3},
    {"pass_direction": "Frente",           "grid_x": 2, "grid_y": 3},
    {"pass_direction": "Frente-Direita",   "grid_x": 3, "grid_y": 3},
    {"pass_direction": "Esquerda",         "grid_x": 1, "grid_y": 2},
    {"pass_direction": "Trás-Esquerda",    "grid_x": 1, "grid_y": 1},
    {"pass_direction": "Trás",             "grid_x": 2, "grid_y": 1},
    {"pass_direction": "Trás-Direita",     "grid_x": 3, "grid_y": 1},
    {"pass_direction": "Direita",          "grid_x": 3, "grid_y": 2},
]

df_grid = pd.DataFrame(grid)
df_grid.to_excel("Grelha Direções.xlsx", index = False)